In [1]:
"""
Carbon Emission Prediction - Global Aggregated (PyTorch)
========================================================
Predicts future CO2 emissions for India using total aggregated
time series data (all sectors combined) with LSTM, GRU, BiLSTM,
CNN-LSTM, and Transformer models.

Input: IND_cleaned_emissions_sources.csv
Output: Forecasts, model comparison tables, and visualization plots.
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
import warnings
import copy
from datetime import datetime

warnings.filterwarnings('ignore')

# Deep Learning imports (PyTorch)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# CONFIGURATION
# ============================================================
INPUT_FILE = 'IND_cleaned_emissions_sources.csv'
OUTPUT_DIR = 'prediction_results_global'
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')
TABLES_DIR = os.path.join(OUTPUT_DIR, 'tables')
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')

# Model hyperparameters
SEQUENCE_LENGTH = 6          # Number of past time steps to use
FORECAST_HORIZON = 6         # Number of future months to predict
EPOCHS = 150
BATCH_SIZE = 16
LEARNING_RATE = 0.001
PATIENCE = 20                # Early stopping patience
VALIDATION_SPLIT = 0.2
TRAIN_END_DATE = '2024-01-01'  # Train: 2021-01 to 2023-12, Test: 2024-01 onward

# Plot settings
plt.style.use("seaborn-v0_8")
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_palette("viridis")

# Create output directories
for d in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("=" * 70)
print("CARBON EMISSION PREDICTION - GLOBAL AGGREGATED (PyTorch)")
print("=" * 70)
print(f"Input file: {INPUT_FILE}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print()

# ============================================================
# 1. DATA LOADING AND GLOBAL AGGREGATION
# ============================================================
print("[1/7] Loading and aggregating data (all sectors combined)...")

df = pd.read_csv(INPUT_FILE)
print(f"  Raw data shape: {df.shape}")
print(f"  Sectors: {df['sector'].nunique()} -> Aggregating all into one series")
print(f"  Year range: {df['year'].min()} - {df['year'].max()}")
print(f"  Months: {df['month'].nunique()}")

# Parse start_time as datetime
df['start_time'] = pd.to_datetime(df['start_time'])

# Global aggregation: group by start_time (monthly), sum emissions across all sectors
# This matches the notebook approach: df_final.groupby('start_time')['emissions_quantity'].sum()
global_monthly = df.groupby('start_time').agg(
    total_emissions=('emissions_quantity', 'sum'),
    mean_emissions=('emissions_quantity', 'mean'),
    source_count=('source_id', 'nunique'),
    mean_activity=('activity', 'mean'),
    mean_emissions_factor=('emissions_factor', 'mean'),
).reset_index()

global_monthly = global_monthly.rename(columns={'start_time': 'date'})
global_monthly = global_monthly.sort_values('date').reset_index(drop=True)
global_monthly['year'] = global_monthly['date'].dt.year
global_monthly['month'] = global_monthly['date'].dt.month

print(f"  Aggregated monthly records: {len(global_monthly)}")
print(f"  Date range: {global_monthly['date'].min()} to {global_monthly['date'].max()}")
print(f"  Train period: 2021-01 to 2023-12 (< {TRAIN_END_DATE})")
print(f"  Test period: {TRAIN_END_DATE} onward")

# Save aggregated data
global_monthly.to_csv(os.path.join(TABLES_DIR, 'global_aggregated_monthly.csv'), index=False)

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
print("\n[2/7] Feature engineering...")

global_monthly['month_sin'] = np.sin(2 * np.pi * global_monthly['month'] / 12)
global_monthly['month_cos'] = np.cos(2 * np.pi * global_monthly['month'] / 12)
global_monthly['year_norm'] = (global_monthly['year'] - global_monthly['year'].min()) / \
    max((global_monthly['year'].max() - global_monthly['year'].min()), 1)
global_monthly['log_emissions'] = np.log1p(global_monthly['total_emissions'].clip(lower=0))

# Rolling features (computed on the full series for context)
global_monthly['rolling_mean_3'] = global_monthly['total_emissions'].rolling(3, min_periods=1).mean()
global_monthly['rolling_std_3'] = global_monthly['total_emissions'].rolling(3, min_periods=1).std().fillna(0)
global_monthly['pct_change'] = global_monthly['total_emissions'].pct_change().fillna(0)

# Feature columns for model input
FEATURE_COLS = ['total_emissions', 'source_count', 'mean_activity',
                'mean_emissions_factor', 'month_sin', 'month_cos', 'year_norm',
                'rolling_mean_3', 'rolling_std_3', 'pct_change']

print(f"  Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"  Target: total_emissions")

# ============================================================
# 3. DATA PREPARATION
# ============================================================
print("\n[3/7] Preparing sequences for DL models...")


def prepare_train_sequences(data, feature_cols, target_col, seq_length, train_end_date):
    """Create training sequences only (data < train_end_date).
    For test evaluation, we use recursive forecasting instead."""
    features = data[feature_cols].values
    target = data[target_col].values.reshape(-1, 1)
    dates = data['date'].values
    split_date = pd.Timestamp(train_end_date)

    # Scale on ALL data to have consistent scaling
    feature_scaler = MinMaxScaler(feature_range=(0, 1))
    target_scaler = MinMaxScaler(feature_range=(0, 1))

    features_scaled = feature_scaler.fit_transform(features)
    target_scaled = target_scaler.fit_transform(target)

    # Create sequences from training period only
    X_train, y_train = [], []
    for i in range(seq_length, len(features_scaled)):
        if dates[i] < split_date:
            X_train.append(features_scaled[i - seq_length:i])
            y_train.append(target_scaled[i, 0])

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    # Get actual test values (>= split_date) for comparison with forecasts
    test_mask = dates >= split_date
    test_actuals = data.loc[test_mask, target_col].values
    test_dates = data.loc[test_mask, 'date'].values

    # Last training sequence (to start recursive forecasting from)
    train_data = data[data['date'] < split_date]
    last_train_seq = feature_scaler.transform(
        train_data[feature_cols].values[-seq_length:]
    )

    return X_train, y_train, test_actuals, test_dates, last_train_seq, feature_scaler, target_scaler


X_train, y_train, test_actuals, test_dates, last_train_seq, feat_scaler, target_scaler = \
    prepare_train_sequences(global_monthly, FEATURE_COLS, 'total_emissions', SEQUENCE_LENGTH, TRAIN_END_DATE)
norm_mask=0.01
n_test_months = len(test_actuals)
print(f"  Train shape: X={X_train.shape}, y={y_train.shape}")
print(f"  Test months (actuals for comparison): {n_test_months}")
print(f"  Test period: {pd.Timestamp(test_dates[0]).strftime('%Y-%m')} to {pd.Timestamp(test_dates[-1]).strftime('%Y-%m')}")
print(f"  Sequence length: {SEQUENCE_LENGTH}, Features: {X_train.shape[2]}")

# ============================================================
# 4. MODEL ARCHITECTURES (PyTorch)
# ============================================================
print("\n[4/7] Defining DL model architectures...")


class LSTMModel(nn.Module):
    """Stacked LSTM model."""
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(hidden_dim, hidden_dim // 2, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)
        self.fc1 = nn.Linear(hidden_dim // 2, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, _ = self.lstm2(x)
        x = self.dropout2(x[:, -1, :])
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


class GRUModel(nn.Module):
    """Stacked GRU model."""
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.gru1 = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)
        self.gru2 = nn.GRU(hidden_dim, hidden_dim // 2, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)
        self.fc1 = nn.Linear(hidden_dim // 2, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x, _ = self.gru1(x)
        x = self.dropout1(x)
        x, _ = self.gru2(x)
        x = self.dropout2(x[:, -1, :])
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


class BiLSTMModel(nn.Module):
    """Bidirectional LSTM model."""
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.bilstm1 = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(0.2)
        self.bilstm2 = nn.LSTM(hidden_dim * 2, hidden_dim // 2, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(0.2)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x, _ = self.bilstm1(x)
        x = self.dropout1(x)
        x, _ = self.bilstm2(x)
        x = self.dropout2(x[:, -1, :])
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


class CNNLSTMModel(nn.Module):
    """CNN-LSTM hybrid model."""
    def __init__(self, input_dim, seq_len, filters=64, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, filters, kernel_size=2)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(filters, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = self.dropout(x[:, -1, :])
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


class TransformerModel(nn.Module):
    """Transformer-based time series model."""
    def __init__(self, input_dim, seq_len, d_model=64, nhead=2, num_layers=2, ff_dim=128):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff_dim,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc1 = nn.Linear(d_model, 64)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.pos_encoding
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


def build_model(model_name, input_dim, seq_len):
    """Factory function to build models by name."""
    if model_name == 'LSTM':
        return LSTMModel(input_dim).to(DEVICE)
    elif model_name == 'GRU':
        return GRUModel(input_dim).to(DEVICE)
    elif model_name == 'BiLSTM':
        return BiLSTMModel(input_dim).to(DEVICE)
    elif model_name == 'CNN-LSTM':
        return CNNLSTMModel(input_dim, seq_len).to(DEVICE)
    elif model_name == 'Transformer':
        return TransformerModel(input_dim, seq_len).to(DEVICE)


MODEL_NAMES = ['LSTM', 'GRU', 'BiLSTM', 'CNN-LSTM']
print(f"  Models defined: {MODEL_NAMES}")

# ============================================================
# 5. MODEL TRAINING AND EVALUATION
# ============================================================
print("\n[5/7] Training all models on global aggregated emissions...")


def train_model(X_train, y_train, target_scaler, model_name):
    """Train a PyTorch model on training data only."""
    input_dim = X_train.shape[2]
    seq_len = X_train.shape[1]
    model = build_model(model_name, input_dim, seq_len)

    # Convert to tensors
    X_tr_tensor = torch.FloatTensor(X_train).to(DEVICE)
    y_tr_tensor = torch.FloatTensor(y_train).reshape(-1, 1).to(DEVICE)

    # Validation split (last 20% of training data for internal validation)
    val_size = max(int(len(X_tr_tensor) * VALIDATION_SPLIT), 1)
    X_val = X_tr_tensor[-val_size:]
    y_val = y_tr_tensor[-val_size:]
    X_tr_tensor = X_tr_tensor[:-val_size]
    y_tr_tensor = y_tr_tensor[:-val_size]

    # DataLoader
    train_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    # Optimizer and loss
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    criterion = nn.MSELoss()

    # Training loop with early stopping
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {'loss': [], 'val_loss': [], 'mae': [], 'val_mae': []}

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        epoch_mae = 0
        n_batches = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            epoch_mae += torch.mean(torch.abs(pred - y_batch)).item()
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        avg_train_mae = epoch_mae / n_batches

        # Validation
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()
            val_mae = torch.mean(torch.abs(val_pred - y_val)).item()

        history['loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['mae'].append(avg_train_mae)
        history['val_mae'].append(val_mae)

        scheduler.step(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history


def forecast_future(model, last_sequence, feature_scaler, target_scaler, n_steps):
    """Generate multi-step future predictions using recursive forecasting."""
    predictions = []
    current_seq = last_sequence.copy()
    model.eval()

    for step in range(n_steps):
        with torch.no_grad():
            input_tensor = torch.FloatTensor(current_seq).unsqueeze(0).to(DEVICE)
            pred_scaled = model(input_tensor).cpu().numpy()[0, 0]
        pred_actual = target_scaler.inverse_transform([[pred_scaled]])[0, 0]
        predictions.append(pred_actual)

        # Update sequence: shift and append prediction
        new_row = current_seq[-1].copy()
        new_features = feature_scaler.inverse_transform(new_row.reshape(1, -1))[0]
        new_features[0] = pred_actual  # total_emissions is first feature
        new_row_scaled = feature_scaler.transform(new_features.reshape(1, -1))[0]
        current_seq = np.vstack([current_seq[1:], new_row_scaled])

    return np.array(predictions)


def robust_mape(y_true, y_pred):
    """MAPE computed only on non-zero actuals."""
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return (np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))) * 100


# Train all models and evaluate via recursive forecast on test period
results = {}
print(f"\n  {'Model':<15} {'Forecast MAPE%':>14} {'Epochs':>8}")
print("  " + "-" * 70)

for model_name in MODEL_NAMES:
    print(f"  Training {model_name}...", end=" ", flush=True)

    # Train on data < 2024-01-01
    model, history = train_model(X_train, y_train, target_scaler, model_name)

    # Recursive forecast from last training sequence for entire test period
    test_forecast = forecast_future(model, last_train_seq, feat_scaler, target_scaler, n_test_months)

    # Compute metrics: forecast vs actual (real test MAPE)
    mae = mean_absolute_error(test_actuals, test_forecast)
    rmse = np.sqrt(mean_squared_error(test_actuals, test_forecast))
    mape = robust_mape(test_actuals, test_forecast)
    r2 = r2_score(test_actuals, test_forecast)

    metrics = {
        'model': model_name,
        'test_mae': mae,
        'test_rmse': rmse,
        'test_r2': r2,
        'test_mape': mape,
        'epochs_trained': len(history['loss']),
        'final_train_loss': history['loss'][-1],
        'final_val_loss': history['val_loss'][-1],
    }

    results[model_name] = {
        'model': model,
        'history': history,
        'metrics': metrics,
        'y_test_actual': test_actuals,
        'y_test_pred': test_forecast,
    }
    print(f"MAPE={mape:.2f}%  Epochs={len(history['loss'])}")

# Identify best model
best_model_name = min(results, key=lambda k: results[k]['metrics']['test_mape'])
print(f"\n  >>> Best Model: {best_model_name} (lowest recursive forecast MAPE)")

# ============================================================
# 6. FUTURE FORECASTING (beyond available data)
# ============================================================
print("\n[6/7] Generating future forecasts beyond available data...")

# Forecast beyond the last available date using best model
last_seq = feat_scaler.transform(global_monthly[FEATURE_COLS].values[-SEQUENCE_LENGTH:])
best_model = results[best_model_name]['model']

forecast = forecast_future(best_model, last_seq, feat_scaler, target_scaler, FORECAST_HORIZON)

# Forecast dates (beyond end of data)
last_date = global_monthly['date'].max()
forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=FORECAST_HORIZON, freq='MS')

forecast_df = pd.DataFrame({
    'date': forecast_dates,
    'forecasted_emissions': forecast,
    'model': best_model_name
})
forecast_df.to_csv(os.path.join(TABLES_DIR, 'global_emission_forecasts.csv'), index=False)

print(f"  Future forecast using {best_model_name} for next {FORECAST_HORIZON} months (beyond data):")
for _, row in forecast_df.iterrows():
    print(f"    {row['date'].strftime('%Y-%m')}: {row['forecasted_emissions']:.4e} tonnes CO2")

# Also forecast with all models for comparison
all_model_forecasts = {}
for mname, mdata in results.items():
    f = forecast_future(mdata['model'], last_seq, feat_scaler, target_scaler, FORECAST_HORIZON)
    all_model_forecasts[mname] = f

# ============================================================
# 7. VISUALIZATION AND REPORTING
# ============================================================
print("\n[7/7] Generating visualizations and reports...")

# --- Figure 1: Model Comparison ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

model_names_list = list(results.keys())
test_maes = [results[m]['metrics']['test_mae'] for m in model_names_list]
test_rmses = [results[m]['metrics']['test_rmse'] for m in model_names_list]
test_r2s = [results[m]['metrics']['test_r2'] for m in model_names_list]
test_mapes = [results[m]['metrics']['test_mape'] for m in model_names_list]

colors = sns.color_palette("viridis", len(model_names_list))

axes[0, 0].barh(model_names_list, test_maes, color=colors)
axes[0, 0].set_xlabel('MAE')
axes[0, 0].set_title('Mean Absolute Error (Test)')

axes[0, 1].barh(model_names_list, test_rmses, color=colors)
axes[0, 1].set_xlabel('RMSE')
axes[0, 1].set_title('Root Mean Squared Error (Test)')

axes[1, 0].barh(model_names_list, test_r2s, color=colors)
axes[1, 0].set_xlabel('R² Score')
axes[1, 0].set_title('R² Score (Test)')

axes[1, 1].barh(model_names_list, test_mapes, color=colors)
axes[1, 1].set_xlabel('MAPE (%)')
axes[1, 1].set_title('Mean Absolute Percentage Error (Test)')

plt.suptitle('Model Comparison - Global Aggregated Emissions', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_comparison.png'), bbox_inches='tight')
plt.close()

# --- Figure 2: Training History (All Models) ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, mname in enumerate(MODEL_NAMES):
    ax = axes[idx]
    h = results[mname]['history']
    ax.plot(h['loss'], label='Train Loss', linewidth=1.5)
    ax.plot(h['val_loss'], label='Val Loss', linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.set_title(f'{mname} (epochs={len(h["loss"])})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

if len(axes) > len(MODEL_NAMES):
    axes[-1].axis('off')

plt.suptitle('Training History - All Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'training_history_all.png'), bbox_inches='tight')
plt.close()

# --- Figure 3: Actual vs Predicted on Test Set (All Models) ---
fig, ax = plt.subplots(figsize=(14, 7))

# Use test_dates from prepare_train_sequences
test_dates_plot = test_dates

ax.plot(test_dates_plot, results[best_model_name]['y_test_actual'], 'k-o',
        label='Actual', markersize=7, linewidth=2, zorder=10)

line_styles = [('b', '--', 's'), ('r', '-.', '^'), ('g', '--', 'D'), ('m', ':', 'v'), ('c', '--', 'p')]
for idx, mname in enumerate(MODEL_NAMES):
    color, ls, marker = line_styles[idx]
    ax.plot(test_dates_plot, results[mname]['y_test_pred'], color=color,
            linestyle=ls, marker=marker, markersize=5, label=f'{mname}', alpha=0.8)

ax.set_xlabel('Date')
ax.set_ylabel('Total CO2 Emissions (tonnes)')
ax.set_title(f'Recursive Forecast vs Actuals: All Models (Train: 2021-2023, Test: 2024+)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'actual_vs_predicted_all_models.png'), bbox_inches='tight')
plt.close()

# --- Figure 4: Best Model - Train + Test + Forecast ---
fig, ax = plt.subplots(figsize=(16, 7))

# Full historical
ax.plot(global_monthly['date'], global_monthly['total_emissions'], 'b-o',
        label='Historical (Actual)', markersize=4, linewidth=1.5)

# Test predictions overlay (recursive forecast on test period)
ax.plot(test_dates_plot, results[best_model_name]['y_test_pred'], 'g--s',
        label=f'Recursive Forecast ({best_model_name})', markersize=6, linewidth=2)

# Forecast
ax.plot(forecast_dates, forecast, 'r--^',
        label=f'Forecast ({best_model_name})', markersize=8, linewidth=2)

# Confidence interval
y_actual_test = results[best_model_name]['y_test_actual']
y_pred_test = results[best_model_name]['y_test_pred']
std_err = np.std(y_actual_test - y_pred_test)
ax.fill_between(forecast_dates,
                forecast - 1.96 * std_err,
                forecast + 1.96 * std_err,
                alpha=0.15, color='red', label='95% CI')

ax.axvline(x=global_monthly['date'].max(), color='gray', linestyle=':', alpha=0.7, label='Forecast Start')
ax.axvline(x=pd.Timestamp(TRAIN_END_DATE), color='orange', linestyle='--', alpha=0.7, label='Train/Test Split')

ax.set_xlabel('Date')
ax.set_ylabel('Total CO2 Emissions (tonnes)')
ax.set_title('India CO2 Emissions: Full Timeline (Historical + Test Predictions + Forecast)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'full_timeline_forecast.png'), bbox_inches='tight')
plt.close()

# --- Figure 5: Multi-Model Forecast Comparison ---
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(global_monthly['date'].tail(12), global_monthly['total_emissions'].tail(12),
        'k-o', label='Historical (last 12 months)', markersize=5, linewidth=1.5)

for idx, mname in enumerate(MODEL_NAMES):
    color, ls, marker = line_styles[idx]
    ax.plot(forecast_dates, all_model_forecasts[mname], color=color,
            linestyle=ls, marker=marker, markersize=6, label=f'{mname} Forecast')

ax.axvline(x=global_monthly['date'].max(), color='gray', linestyle=':', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Total CO2 Emissions (tonnes)')
ax.set_title('Future Forecast Comparison: All Models')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'forecast_all_models.png'), bbox_inches='tight')
plt.close()

# --- Figure 6: Residual Analysis (Best Model) ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

residuals = y_actual_test - y_pred_test

axes[0].plot(test_dates_plot, residuals, 'b-o', markersize=5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Residual')
axes[0].set_title(f'{best_model_name} - Residuals Over Time')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

axes[1].hist(residuals, bins=10, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_pred_test, residuals, alpha=0.7, s=50, color='steelblue')
axes[2].axhline(y=0, color='r', linestyle='--')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residuals vs Predicted')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Residual Analysis - {best_model_name}', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'residual_analysis.png'), bbox_inches='tight')
plt.close()

# --- Figure 7: Scatter Plot - Actual vs Predicted (Best Model) ---
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_actual_test, y_pred_test, alpha=0.8, s=80, color='steelblue', edgecolor='black', linewidth=0.5)
min_val = min(y_actual_test.min(), y_pred_test.min())
max_val = max(y_actual_test.max(), y_pred_test.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Emissions')
ax.set_ylabel('Predicted Emissions')
ax.set_title(f'{best_model_name}: Actual vs Predicted (R²={results[best_model_name]["metrics"]["test_r2"]:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'scatter_actual_vs_predicted.png'), bbox_inches='tight')
plt.close()

# ============================================================
# RESULTS TABLES AND SUMMARY REPORT
# ============================================================

# Model comparison table
model_comparison = pd.DataFrame([results[m]['metrics'] for m in results])
model_comparison = model_comparison.sort_values('test_rmse')
model_comparison.to_csv(os.path.join(TABLES_DIR, 'model_comparison.csv'), index=False)

# All model forecasts table
forecast_comparison = pd.DataFrame({'date': forecast_dates})
for mname in MODEL_NAMES:
    forecast_comparison[f'{mname}_forecast'] = all_model_forecasts[mname]
forecast_comparison.to_csv(os.path.join(TABLES_DIR, 'forecast_all_models.csv'), index=False)

# Summary report
summary_lines = [
    "=" * 70,
    "CARBON EMISSION PREDICTION - GLOBAL AGGREGATED SUMMARY",
    "=" * 70,
    f"Date Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Input Data: {INPUT_FILE}",
    f"Data Period: {global_monthly['date'].min()} to {global_monthly['date'].max()}",
    f"Total Raw Records: {len(df):,}",
    f"Monthly Aggregated Points: {len(global_monthly)}",
    f"Train Period: 2021-01 to 2023-12 (< {TRAIN_END_DATE})",
    f"Test Period: {TRAIN_END_DATE} onward ({n_test_months} months)",
    f"Train sequences: {X_train.shape[0]}",
    f"Test evaluation: Recursive forecast for {n_test_months} months",
    "",
    "-" * 70,
    "MODEL PERFORMANCE COMPARISON (Recursive Forecast MAPE)",
    "-" * 70,
    f"{'Model':<15} {'MAPE%':>8}",
    "-" * 70,
]

for _, row in model_comparison.iterrows():
    summary_lines.append(
        f"{row['model']:<15} {row['test_mape']:>8.2f}"
    )

summary_lines.extend([
    "",
    f">>> Best Model: {best_model_name} (lowest recursive forecast MAPE)",
    f"    Test MAPE: {results[best_model_name]['metrics']['test_mape']:.2f}%",
    "",
    "-" * 70,
    f"FORECAST (Next {FORECAST_HORIZON} Months) - Best Model: {best_model_name}",
    "-" * 70,
])

for _, row in forecast_df.iterrows():
    summary_lines.append(f"  {row['date'].strftime('%Y-%m')}: {row['forecasted_emissions']:.4e} tonnes CO2")

summary_lines.extend([
    "",
    "-" * 70,
    "CONFIGURATION",
    "-" * 70,
    f"Sequence Length: {SEQUENCE_LENGTH}",
    f"Forecast Horizon: {FORECAST_HORIZON} months",
    f"Features: {FEATURE_COLS}",
    f"Max Epochs: {EPOCHS}",
    f"Batch Size: {BATCH_SIZE}",
    f"Learning Rate: {LEARNING_RATE}",
    f"Early Stopping Patience: {PATIENCE}",
    f"Validation Split: {VALIDATION_SPLIT}",
    f"Train Period: 2021-01 to 2023-12 | Test: {TRAIN_END_DATE} onward",
    "",
    "-" * 70,
    "OUTPUT FILES",
    "-" * 70,
    f"Tables: {TABLES_DIR}/",
    f"  - model_comparison.csv",
    f"  - global_emission_forecasts.csv",
    f"  - forecast_all_models.csv",
    f"  - global_aggregated_monthly.csv",
    f"Figures: {FIGURES_DIR}/",
    f"  - model_comparison.png",
    f"  - training_history_all.png",
    f"  - actual_vs_predicted_all_models.png",
    f"  - full_timeline_forecast.png",
    f"  - forecast_all_models.png",
    f"  - residual_analysis.png",
    f"  - scatter_actual_vs_predicted.png",
    "=" * 70,
])

summary_text = '\n'.join(summary_lines)
with open(os.path.join(TABLES_DIR, 'prediction_summary_report.txt'), 'w') as f:
    f.write(summary_text)

print("\n" + summary_text)

print("\n\n[DONE] All outputs saved to:", OUTPUT_DIR)
print(f"  Tables: {len(os.listdir(TABLES_DIR))} files")
print(f"  Figures: {len(os.listdir(FIGURES_DIR))} files")


CARBON EMISSION PREDICTION - GLOBAL AGGREGATED (PyTorch)
Input file: IND_cleaned_emissions_sources.csv
PyTorch version: 1.13.1
Device: cpu
CUDA Available: False

[1/7] Loading and aggregating data (all sectors combined)...
  Raw data shape: (1439021, 20)
  Sectors: 7 -> Aggregating all into one series
  Year range: 2021 - 2025
  Months: 12
  Aggregated monthly records: 54
  Date range: 2021-01-01 00:00:00 to 2025-06-01 00:00:00
  Train period: 2021-01 to 2023-12 (< 2024-01-01)
  Test period: 2024-01-01 onward

[2/7] Feature engineering...
  Features (10): ['total_emissions', 'source_count', 'mean_activity', 'mean_emissions_factor', 'month_sin', 'month_cos', 'year_norm', 'rolling_mean_3', 'rolling_std_3', 'pct_change']
  Target: total_emissions

[3/7] Preparing sequences for DL models...
  Train shape: X=(30, 6, 10), y=(30,)
  Test months (actuals for comparison): 18
  Test period: 2024-01 to 2025-06
  Sequence length: 6, Features: 10

[4/7] Defining DL model architectures...
  Models d

In [2]:
print("Done")

Done
